In [6]:
import pandas as pd

# Load datasets
movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

# Shape (rows, columns)
print("Movies Shape:", movies.shape)
print("Ratings Shape:", ratings.shape)

print()

# Column names
print("Movies Columns:")
print(movies.columns)

print()

print("Ratings Columns:")
print(ratings.columns)

print()

# Information about datasets
print("Movies Info")
print(movies.info())

print()

print("Ratings Info")
print(ratings.info())

Movies Shape: (9742, 3)
Ratings Shape: (100836, 4)

Movies Columns:
Index(['movieId', 'title', 'genres'], dtype='str')

Ratings Columns:
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')

Movies Info
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 626.1 KB
None

Ratings Info
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
None


In [ ]:
print("Missing Values in Movies")
print(movies.isnull().sum())

print()

print("Missing Values in Ratings")
print(ratings.isnull().sum())

print()

# -------------------------------
# Duplicate Rows
# -------------------------------

print("Duplicate Movies:", movies.duplicated().sum())
print("Duplicate Ratings:", ratings.duplicated().sum())

movie_data=pd.merge(movies,ratings , on='movieId')

print("Merged Data Shape:", movie_data.shape)
print(movie_data.head())


movie_ratings=movie_data.groupby('title')["rating"].mean().sort_values(ascending=False)

print()

print(movie_ratings.head(10))


ratings_count=movie_data.groupby('title')["rating"].count().sort_values(ascending=False)




Missing Values in Movies
movieId    0
title      0
genres     0
dtype: int64

Missing Values in Ratings
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Duplicate Movies: 0
Duplicate Ratings: 0
Merged Data Shape: (100836, 6)
   movieId             title                                       genres  \
0        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
1        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
2        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
3        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
4        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   

   userId  rating   timestamp  
0       1     4.0   964982703  
1       5     4.0   847434962  
2       7     4.5  1106635946  
3      15     2.5  1510577970  
4      17     4.5  1305696483  

title
Karlson Returns (1970)                                                         5.0
Zeitg

In [15]:
recommendation=pd.DataFrame()


recommendation["Average Rating"]=movie_ratings
recommendation["Rating Count"]=ratings_count

print(recommendation.head(10))

                                                    Average Rating  \
title                                                                
Karlson Returns (1970)                                         5.0   
Zeitgeist: Moving Forward (2011)                               5.0   
Dream of Light (a.k.a. Quince Tree Sun, The) (S...             5.0   
Dragons: Gift of the Night Fury (2011)                         5.0   
12 Angry Men (1997)                                            5.0   
Justice League: Doom (2012)                                    5.0   
Junior and Karlson (1968)                                      5.0   
Jump In! (2007)                                                5.0   
Human Condition III, The (Ningen no joken III) ...             5.0   
Louis Theroux: Law & Disorder (2008)                           5.0   

                                                    Rating Count  
title                                                             
Karlson Returns (1970)   

In [16]:
recommendation = recommendation[
    recommendation["Rating Count"] >= 50
]

# ============================
# Sort by Average Rating
# ============================

recommendation = recommendation.sort_values(
    by="Average Rating",
    ascending=False
)

# ============================
# Top 10 Movies
# ============================

print("\nTop 10 Recommended Movies")
print(recommendation.head(10))


Top 10 Recommended Movies
                                                    Average Rating  \
title                                                                
Shawshank Redemption, The (1994)                          4.429022   
Godfather, The (1972)                                     4.289062   
Fight Club (1999)                                         4.272936   
Cool Hand Luke (1967)                                     4.271930   
Dr. Strangelove or: How I Learned to Stop Worry...        4.268041   
Rear Window (1954)                                        4.261905   
Godfather: Part II, The (1974)                            4.259690   
Departed, The (2006)                                      4.252336   
Goodfellas (1990)                                         4.250000   
Casablanca (1942)                                         4.240000   

                                                    Rating Count  
title                                                            

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies["genres"])
print(tfidf_matrix.shape)
print(tfidf.get_feature_names_out())

(9742, 23)
['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'genres' 'horror' 'imax'
 'listed' 'musical' 'mystery' 'noir' 'romance' 'sci' 'thriller' 'war'
 'western']


In [22]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tfidf_matrix)

print(similarity.shape)

(9742, 9742)


In [44]:
def recommend_movies(movie_name):
    
    # Check whether the movie exists
    if movie_name not in movies["title"].values:
        print("Movie not found!")
        return

    movie_index = movies[movies["title"] == movie_name].index[0]

    similarity_scores = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(similarity_scores)),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    print("\nRecommended Movies:\n")

    for movie_index, score in movie_list:
        print(f"{movies.iloc[movie_index].title}  -->  Similarity : {score:.3f}")

In [47]:
movie = input("Enter Movie Name: ")

recommend_movies(movie)

Movie not found!
